# Mat & Comp – ML 2: Regresión lineal
## La mejor recta: tu primer modelo de machine learning

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 1

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar qué es un **modelo** y qué significa **entrenarlo**.
2. Medir qué tan buena es una recta con el **error cuadrático medio** (ECM).
3. Encontrar la mejor recta con el método de **mínimos cuadrados**, programado a mano con numpy.
4. Entrenar el mismo modelo con **scikit-learn** y usarlo para **predecir**.
5. Evaluar el modelo con el **coeficiente R²** y reconocer los peligros de **extrapolar**.

## 🔍 ¿Qué nota sacará Mateo?

En la lección anterior vimos que en nuestros 60 estudiantes las horas de estudio y la nota tienen una correlación cercana a **0.9**: los puntos casi forman una recta.

Llega un estudiante nuevo, Mateo, y nos cuenta que estudia **9 horas a la semana**.

✏️ *Mira la gráfica de abajo y predice: ¿qué nota crees que tiene Mateo? ¿Cómo justificarías tu respuesta?*

**Mi predicción:**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
df = pd.read_csv(url)

x = df["horas_estudio"].to_numpy()        # lo que conocemos (entrada)
y = df["nota_matematicas"].to_numpy()     # lo que queremos predecir (salida)

plt.scatter(x, y, alpha=0.7)
plt.axvline(9, color="gray", linestyle=":", label="Mateo: 9 horas")
plt.xlabel("horas de estudio a la semana")
plt.ylabel("nota de matemáticas")
plt.title("¿Qué nota tendrá Mateo?")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 1. Un modelo es una regla con perillas

Para predecir, proponemos una regla sencilla: una **recta**.

$$\text{nota} \approx m \cdot \text{horas} + b$$

- `m` es la **pendiente**: cuánto sube la nota por cada hora extra de estudio.
- `b` es el **intercepto**: la nota que predice la recta para alguien que estudia 0 horas.

`m` y `b` son los **parámetros** del modelo: como dos perillas que se pueden girar. **Entrenar** el modelo significa encontrar los valores de las perillas que mejor se ajustan a los datos.

Así se ve una recta con valores elegidos "a ojo":

In [ ]:
m, b = 0.25, 1.0          # valores inventados

plt.scatter(x, y, alpha=0.7)
xs = np.linspace(0, 14, 100)
plt.plot(xs, m * xs + b, color="red", label=f"nota = {m}·horas + {b}")
plt.xlabel("horas de estudio")
plt.ylabel("nota")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Predicción para Mateo:", m * 9 + b)

¿Es una buena recta? Parece pasar un poco por debajo de los puntos de la izquierda y por encima de los de la derecha. Necesitamos una forma de **medir** qué tan buena es.

## 2. Medir el error

Para cada estudiante, la recta hace una **predicción** `m·x + b`. La diferencia entre la nota real y la predicha se llama **residuo**:

$$\text{residuo} = y_\text{real} - y_\text{predicho}$$

En la gráfica, cada residuo es un segmento vertical entre el punto y la recta.

In [ ]:
m, b = 0.25, 1.0
prediccion = m * x + b
residuos = y - prediccion

plt.scatter(x, y, alpha=0.7, zorder=3)
plt.plot(xs, m * xs + b, color="red")
for xi, yi, pi in zip(x, y, prediccion):
    plt.plot([xi, xi], [yi, pi], color="gray", linewidth=1)
plt.xlabel("horas de estudio")
plt.ylabel("nota")
plt.title("Los residuos: distancia vertical de cada punto a la recta")
plt.grid(alpha=0.3)
plt.show()

print("Primeros 5 residuos:", residuos[:5].round(2))

Para resumir todos los residuos en un solo número no podemos simplemente sumarlos: los positivos y los negativos se cancelarían. La solución más usada es **elevarlos al cuadrado** (así todos son positivos, y los errores grandes pesan más) y sacar el promedio:

$$\text{ECM} = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - (m x_i + b)\right)^2$$

Este es el **error cuadrático medio** (ECM, o *MSE* en inglés). **Mientras más pequeño, mejor la recta.**

In [ ]:
def ecm(m, b):
    prediccion = m * x + b
    return np.mean((y - prediccion) ** 2)

print("ECM de la recta 0.25·x + 1.0:", round(ecm(0.25, 1.0), 4))
print("ECM de la recta 0.20·x + 1.5:", round(ecm(0.20, 1.5), 4))
print("ECM de una recta horizontal en el promedio:", round(ecm(0, y.mean()), 4))

## 3. 🎛️ Juega con las perillas

Mueve los sliders para encontrar la recta con el **ECM más pequeño** que puedas. Anota tu mejor resultado: lo compararemos con la respuesta exacta.

In [ ]:
from ipywidgets import interact, FloatSlider

def dibujar_recta(m=0.25, b=1.0, ver_residuos=True):
    pred = m * x + b
    plt.figure(figsize=(7, 4.5))
    plt.scatter(x, y, alpha=0.7, zorder=3)
    plt.plot(xs, m * xs + b, color="red")
    if ver_residuos:
        for xi, yi, pi in zip(x, y, pred):
            plt.plot([xi, xi], [yi, pi], color="gray", linewidth=1)
    plt.ylim(0, 6)
    plt.xlabel("horas de estudio")
    plt.ylabel("nota")
    plt.title(f"nota = {m:.2f}·horas + {b:.2f}     ECM = {ecm(m, b):.4f}")
    plt.grid(alpha=0.3)
    plt.show()

interact(dibujar_recta,
         m=FloatSlider(value=0.25, min=-0.2, max=0.5, step=0.01, continuous_update=False),
         b=FloatSlider(value=1.0, min=0.0, max=4.0, step=0.05, continuous_update=False));

## 4. El paisaje del error

Cada par de valores `(m, b)` tiene su ECM. Podemos calcularlo para **muchísimos** pares y pintarlo como un mapa: los colores oscuros son errores pequeños. La mejor recta está en el **fondo del valle**.

In [ ]:
valores_m = np.linspace(0.0, 0.4, 120)
valores_b = np.linspace(0.5, 3.0, 120)
mapa = np.zeros((len(valores_b), len(valores_m)))

for i, bb in enumerate(valores_b):
    for j, mm in enumerate(valores_m):
        mapa[i, j] = ecm(mm, bb)

# Fuerza bruta: el punto del mapa con menor error
i_min, j_min = np.unravel_index(mapa.argmin(), mapa.shape)
print(f"Mejor punto de la cuadrícula: m = {valores_m[j_min]:.3f}, b = {valores_b[i_min]:.3f}, ECM = {mapa.min():.4f}")

plt.figure(figsize=(7, 5))
plt.contourf(valores_m, valores_b, np.log(mapa), levels=30, cmap="viridis")
plt.colorbar(label="log(ECM)  (más oscuro = menos error)")
plt.scatter(valores_m[j_min], valores_b[i_min], color="red", marker="*", s=200, label="mínimo")
plt.xlabel("m (pendiente)")
plt.ylabel("b (intercepto)")
plt.title("El paisaje del error")
plt.legend()
plt.show()

Probar 14 400 combinaciones funcionó, pero solo da una respuesta aproximada (depende de la cuadrícula) y con más parámetros sería imposible: con 10 perillas y 120 valores por perilla serían 120¹⁰ combinaciones. Necesitamos algo mejor.

## 5. Mínimos cuadrados: la respuesta exacta

Usando cálculo (derivadas) se puede demostrar que el ECM es mínimo exactamente cuando

$$m = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad\qquad b = \bar{y} - m\,\bar{x}$$

donde $\bar{x}$ y $\bar{y}$ son los promedios. Este es el método de **mínimos cuadrados**, inventado hace más de 200 años por Legendre y Gauss (¡el mismo Gauss de la suma del 1 al 100!).

Programémoslo a mano con numpy:

In [ ]:
x_prom = x.mean()
y_prom = y.mean()

m_mejor = np.sum((x - x_prom) * (y - y_prom)) / np.sum((x - x_prom) ** 2)
b_mejor = y_prom - m_mejor * x_prom

print(f"Mejor recta: nota = {m_mejor:.4f}·horas + {b_mejor:.4f}")
print(f"ECM mínimo: {ecm(m_mejor, b_mejor):.4f}")

¿Qué tan cerca estuviste con los sliders?

**Interpretación:**
- La pendiente dice que **cada hora extra de estudio a la semana se asocia con unos 0.2 puntos (2 décimas) más de nota**.
- El intercepto dice que alguien que no estudia nada tendría, según el modelo, una nota cercana a 1.7.

### 🎯 La nota de Mateo

In [ ]:
horas_mateo = 9
nota_mateo = m_mejor * horas_mateo + b_mejor
print(f"Predicción para Mateo: {nota_mateo:.2f}")

plt.scatter(x, y, alpha=0.6)
plt.plot(xs, m_mejor * xs + b_mejor, color="red", label="mejor recta")
plt.scatter([horas_mateo], [nota_mateo], color="orange", s=150, marker="*", zorder=4, label="Mateo")
plt.xlabel("horas de estudio")
plt.ylabel("nota")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Lo mismo con scikit-learn

**scikit-learn** es la librería de machine learning más usada en Python. Casi todos sus modelos funcionan igual, en tres pasos:

1. **Crear** el modelo: `modelo = LinearRegression()`
2. **Entrenar**: `modelo.fit(X, y)`
3. **Predecir**: `modelo.predict(X_nuevo)`

⚠️ scikit-learn espera que las entradas `X` sean una **tabla** (con filas y columnas), aunque tenga una sola columna. Por eso usamos `df[["horas_estudio"]]`, con **doble** corchete.

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["horas_estudio"]]          # tabla con una columna
modelo = LinearRegression()
modelo.fit(X, y)

print("Pendiente (coef_):   ", modelo.coef_[0].round(4))
print("Intercepto (intercept_):", modelo.intercept_.round(4))

In [ ]:
nuevos = pd.DataFrame({"horas_estudio": [0, 5, 9, 14]})
nuevos["nota_predicha"] = modelo.predict(nuevos).round(2)
nuevos

¡Los mismos números que calculamos a mano! scikit-learn usa por dentro las mismas fórmulas (en una versión más general).

## 7. ¿Qué tan bueno es el modelo? El coeficiente R²

El ECM depende de las unidades. Un número más fácil de interpretar es el **R²**: compara el error de nuestra recta con el error de la "recta más tonta", que siempre predice el promedio.

$$R^2 = 1 - \frac{\text{ECM de nuestra recta}}{\text{ECM de predecir siempre el promedio}}$$

- R² = 1: la recta pasa exactamente por todos los puntos.
- R² = 0: la recta no es mejor que adivinar el promedio.

In [ ]:
r2_a_mano = 1 - ecm(m_mejor, b_mejor) / ecm(0, y.mean())
print("R² a mano:         ", round(r2_a_mano, 4))
print("R² con scikit-learn:", round(modelo.score(X, y), 4))

Las horas de estudio explican cerca del **80 %** de la variación de las notas. El resto se debe a otras cosas: el sueño, las redes, y factores que no medimos.

## 8. Más de una entrada: regresión múltiple

Podemos usar **varias columnas** a la vez. El modelo se vuelve

$$\text{nota} \approx m_1 \cdot \text{estudio} + m_2 \cdot \text{sueño} + m_3 \cdot \text{redes} + b$$

Ya no es una recta sino un "plano" en más dimensiones, pero con scikit-learn el código es casi idéntico.

In [ ]:
df["horas_sueno"] = df["horas_sueno"].fillna(df["horas_sueno"].mean())   # como en ML 1

columnas = ["horas_estudio", "horas_sueno", "horas_redes"]
X3 = df[columnas]
modelo3 = LinearRegression().fit(X3, y)

for col, coef in zip(columnas, modelo3.coef_):
    print(f"{col:15s} {coef:+.3f}")
print(f"{'intercepto':15s} {modelo3.intercept_:+.3f}")
print("R² con una columna:  ", round(modelo.score(X, y), 3))
print("R² con tres columnas:", round(modelo3.score(X3, y), 3))

Cada coeficiente dice cuánto cambia la nota predicha si esa columna aumenta en 1 **y las demás se quedan iguales**. Por ejemplo, una hora más de sueño por noche se asocia con una nota algo mayor, y una hora más de redes al día, con una nota algo menor.

## 9. ⚠️ Cuidado con extrapolar

Un modelo solo es confiable **cerca de los datos con los que se entrenó**. ¿Qué pasa si le preguntamos por alguien que estudia 30 horas?

In [ ]:
print("Predicción para 30 horas:", modelo.predict(pd.DataFrame({"horas_estudio": [30]}))[0].round(2))

¡Una nota mayor que 5.0, que es imposible! Nuestros datos solo llegaban hasta 14 horas, y nada garantiza que la relación siga siendo una recta más allá. **Extrapolar** (predecir fuera del rango de los datos) es uno de los errores más comunes en machine learning.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en el ECM usas el valor absoluto `np.abs(...)` en vez del cuadrado? ¿Cambia mucho la mejor recta? (Pruébalo con el mapa de la sección 4.)
2. …agregas a los datos un estudiante "raro" que estudia 14 horas y saca 1.0? ¿Cuánto cambia la pendiente? (Pista: `np.append(x, 14)`.)
3. …entrenas el modelo con `df["horas_estudio"]` (un solo corchete) en scikit-learn? ¿Qué error aparece?
4. …intercambias los papeles y predices las horas de estudio a partir de la nota? ¿La pendiente es 1/m?
5. …calculas R² para el modelo que usa solo `horas_redes`? ¿Es mayor o menor que el de las horas de estudio?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil).** Usa la recta de mínimos cuadrados para predecir la nota de alguien que estudia 0, 3, 7 y 12 horas. ¿Cuántas horas necesitaría estudiar, según el modelo, para sacar 4.0?

**Ejercicio 2 (fácil).** Calcula el ECM de estas tres rectas y ordénalas de mejor a peor: `0.3·x + 1.0`, `0.2·x + 1.7` y `0.1·x + 2.4`.

**Ejercicio 3 (medio).** Usa las fórmulas de mínimos cuadrados (a mano, con numpy) para encontrar la mejor recta que predice la nota a partir de `horas_redes`. ¿La pendiente es positiva o negativa? ¿Qué significa? Compara con `LinearRegression`.

**Ejercicio 4 (medio). Residuos.** Con la mejor recta de horas de estudio, crea una columna `residuo = nota real − nota predicha`.
1. ¿Qué estudiante tiene el residuo **más grande** (le fue mucho mejor de lo que predice el modelo)? ¿Y el más negativo?
2. Mira sus horas de sueño y de redes. ¿Ayudan a explicar el residuo?
3. Comprueba que el promedio de los residuos es (casi) 0. ¿Por qué crees que pasa?

**Ejercicio 5 (medio).** Entrena un modelo de regresión múltiple con **solo** `horas_sueno` y `horas_redes` (sin las horas de estudio). ¿Cuál es su R²? ¿Qué te dice eso sobre la importancia de cada variable?

## 🏆 Reto opcional: el precio de las casas

Con los datos de California que vienen en Colab (`sample_data/california_housing_train.csv`, como en el reto de ML 1):

1. Entrena una regresión lineal que prediga `median_house_value` a partir de `median_income`. Interpreta la pendiente.
2. Grafica los puntos y la recta. ¿Ves algo raro en la parte de arriba? (Pista: el precio máximo registrado está "cortado" en 500 001).
3. Agrega más columnas al modelo (`housing_median_age`, `total_rooms`, `latitude`, `longitude`…). ¿Cuánto mejora el R²?

## 🎬 Cierre

Hoy aprendiste que:
- Un **modelo** es una regla con parámetros, y **entrenar** es encontrar los parámetros que mejor se ajustan a los datos.
- El **error cuadrático medio** mide qué tan lejos están las predicciones de los valores reales.
- **Mínimos cuadrados** da la fórmula exacta de la mejor recta, y scikit-learn la calcula con `fit`.
- **R²** dice qué parte de la variación explica el modelo, y extrapolar fuera de los datos es peligroso.

**Para pensar:** la recta tenía una fórmula exacta porque es un modelo muy simple. Pero las redes neuronales tienen **millones** de parámetros y no existe ninguna fórmula para encontrarlos. ¿Cómo se entrena entonces un modelo así? Recuerda el paisaje del error de la sección 4: si estuvieras con los ojos vendados en una montaña, ¿cómo bajarías al valle? Esa es la idea del **descenso de gradiente**, el tema de la próxima lección. 🚀